# Gemma-2-9B QLoRA — preference classifier (TRAINING)
Fine-tunes Gemma-2-9B (4-bit QLoRA) as a 3-class classifier and saves the LoRA adapter to `/kaggle/working/adapter`. Enable **GPU** and **Internet** for this notebook, and attach the **Gemma-2-9B** model under *Add Input → Models*.

In [ ]:
# transformers 5.7.0 (needs bitsandbytes>=0.46.1 for 4-bit). Do a Factory reset first,
# then Run All patiently (the first `import transformers` scans package metadata, ~1-3 min).
!pip install -q 'transformers==5.7.0' 'bitsandbytes>=0.46.1'

In [ ]:
# Every huggingface_hub version on this image has a @strict that rejects transformers'
# Gemma2Config (StrictDataclassDefinitionError), and no allowed hub version avoids it.
# Neutralize @strict to a pass-through -- it only skips optional config input-validation,
# nothing that affects the model or training. Must run BEFORE the model is loaded.
import huggingface_hub, huggingface_hub.dataclasses as _hfd
_noop = lambda cls=None, **kw: (cls if cls is not None else (lambda c: c))
_hfd.strict = _noop
huggingface_hub.strict = _noop
print('patched huggingface_hub.strict')

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # reduce fragmentation
import torch, time
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import datasets

MAX_LEN = 512           # short so 9B on T4 x2 finishes 1 epoch well under the 9h cap
SUBSAMPLE = 10000       # ~594 steps -> ~5-6h. Raise once a full run succeeds. None = all rows
EPOCHS = 1
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:

import json, re, os, glob, numpy as np, pandas as pd

TARGETS = ["winner_model_a", "winner_model_b", "winner_tie"]

def find_dir(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"nothing matched {pattern} under /kaggle/input"
    return os.path.dirname(hits[0])

def parse_list(x):
    if isinstance(x, list): return [str(t) for t in x]
    if not isinstance(x, str): return [""]
    try: v = json.loads(x)
    except Exception: return [x]
    if isinstance(v, list): return ["" if t is None else str(t) for t in v]
    return ["" if v is None else str(v)]

def join(x): return "\n".join(parse_list(x))

def build_text(prompt, resp_a, resp_b, max_chars=7000):
    """Structured prompt for the classifier. Truncate each field head+tail."""
    def clip(s, n):
        s = s or ""
        return s if len(s) <= n else s[: n // 2] + " ... " + s[-n // 2 :]
    return (
        "You are judging which chatbot response a human prefers.\n\n"
        "### Prompt\n" + clip(prompt, max_chars // 3) +
        "\n\n### Response A\n" + clip(resp_a, max_chars // 3) +
        "\n\n### Response B\n" + clip(resp_b, max_chars // 3) +
        "\n\n### Which is preferred? A, B, or tie."
    )


In [ ]:
# Locate base model + competition data (attach both via Add Input).
BASE = find_dir('/kaggle/input/**/config.json')
COMP = find_dir('/kaggle/input/**/train.csv')
print('BASE =', BASE, '\nCOMP =', COMP)

In [ ]:
df = pd.read_csv(f'{COMP}/train.csv')
if SUBSAMPLE: df = df.sample(SUBSAMPLE, random_state=42).reset_index(drop=True)
df['text'] = [build_text(join(p), join(a), join(b))
              for p, a, b in zip(df['prompt'], df['response_a'], df['response_b'])]
df['label'] = np.argmax(df[TARGETS].values, axis=1)
tr, va = train_test_split(df, test_size=0.05, stratify=df['label'], random_state=42)
print(len(tr), 'train /', len(va), 'val')

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def to_ds(frame):
    ds = datasets.Dataset.from_pandas(frame[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=['text'])
ds_tr, ds_va = to_ds(tr), to_ds(va)

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
# device_map='auto' shards the 9B across all visible GPUs (e.g. T4 x2 -> ~29GB total),
# which is what makes 9B QLoRA training fit. On a single GPU it just uses that one.
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3, quantization_config=bnb, dtype=torch.float16, device_map='auto')
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)

# Auto-detect linear layers so LoRA works on any Gemma version (2/3/4).
import bitsandbytes as bnb
def linear_names(m):
    names = set()
    for n, mod in m.named_modules():
        if isinstance(mod, (torch.nn.Linear, bnb.nn.Linear4bit)):
            names.add(n.split('.')[-1])
    names -= {'lm_head', 'score', 'classifier'}  # head is trained separately
    return sorted(names)
targets = linear_names(model)
print('LoRA target modules:', targets)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='SEQ_CLS', target_modules=targets)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
def metric(eval_pred):
    logits, labels = eval_pred
    p = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return {'log_loss': log_loss(labels, p, labels=[0,1,2])}

args = TrainingArguments(
    output_dir='/kaggle/working/ckpt', per_device_train_batch_size=1,
    gradient_accumulation_steps=16, per_device_eval_batch_size=1,
    learning_rate=1e-4, num_train_epochs=EPOCHS, warmup_ratio=0.03,
    fp16=True, gradient_checkpointing=True, logging_steps=25,
    eval_strategy='epoch', save_strategy='no', report_to='none', optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=metric)
trainer.train()
print(trainer.evaluate())

In [ ]:
model.save_pretrained('/kaggle/working/adapter')
tok.save_pretrained('/kaggle/working/adapter')
print('saved adapter -> /kaggle/working/adapter')
# Next: Save Version (commit). Then create a Kaggle Dataset from this output,
# and attach it to the gemma-infer notebook.